In [1]:
import lropt

In [16]:
import cvxpy as cp

# import matplotlib.pyplot as plt
import numpy as np
import numpy.random as npr
import numpy.testing as npt
import scipy as sc
import torch
from sklearn.model_selection import train_test_split

In [17]:
n = 2
kappa = -0.01
seed = 15
np.random.seed(seed)
dist = (np.array([25, 10, 60, 50, 40, 30, 30, 20,
                  20, 15, 10, 10, 10, 10, 10, 10])/10)[:n]
y_data = np.random.dirichlet(dist, 10)
y = lropt.Parameter(n, data=y_data)

In [18]:
def gen_demand_intro(N, seed):
    np.random.seed(seed)
    sig = np.array([[0.5, -0.3], [-0.3, 0.4]])
    mu = np.array((0.3, 0.3))
    d_train = np.random.multivariate_normal(mu, sig, N)
    # d_train = np.exp(d_train)
    return d_train

In [19]:
data = gen_demand_intro(600, seed=15)
u = lropt.UncertainParameter(n,uncertainty_set=lropt.Ellipsoidal(p=2,data=data))
# Formulate the Robust Problem
x = cp.Variable(n)
t = cp.Variable()

In [20]:
objective = cp.Minimize(t + 0.2*cp.norm(x - y, 1))
constraints = [-x@u <= t, cp.sum(x) == 1, x >= 0]
eval_exp = -x @ u + 0.2*cp.norm(x-y, 1)
prob = lropt.RobustProblem(objective, constraints, eval_exp=eval_exp)
test_p = 0.1
s = 5
train, _ = train_test_split(data, test_size=int(
    data.shape[0]*test_p), random_state=s)
init = sc.linalg.sqrtm(sc.linalg.inv(np.cov(train.T)))
-init@np.mean(train, axis=0)
np.random.seed(15)
initn = np.random.rand(n, n) + 0.1*init + 0.5*np.eye(n)
init_bvaln = -initn@(np.mean(train, axis=0) - 0.3*np.ones(n))

In [21]:
# Train A and b
result = prob.train(lr=0.01, num_iter=100, momentum=0.8,
                    optimizer="SGD",
                    seed=s, init_A=initn, init_b=init_bvaln,
                    init_lam=1, init_mu=1,
                    mu_multiplier=1.01, init_alpha=0., test_percentage=test_p, kappa=kappa,
                    n_jobs=8, random_init=True, num_random_init=2, position=True)


run 1: test value N/A, violations N/A:   0%|          | 0/100 [00:00<?, ?it/s]/Users/annieliang/anaconda3/envs/lropt/lib/python3.11/site-packages/threadpoolctl.py:1010: RuntimeWarning: 
Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this
can cause random crashes or deadlocks on Linux when loaded in the
same Python program.
Using threadpoolctl may cause crashes or deadlocks. For more
information and possible workarounds, please see
    https://github.com/joblib/threadpoolctl/blob/master/multiple_openmp.md

  warnings.warn(msg, RuntimeWarning)
/Users/annieliang/anaconda3/envs/lropt/lib/python3.11/site-packages/threadpoolctl.py:1010: RuntimeWarning: 
Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this
can cause random crashes or deadlocks on Linux when loaded in the
same Python program.
Using threadpoolctl may cause crashe

In [22]:
result.df

,step,Lagrangian_val,Train_val,Probability_violations_train,Violations_train,A_norm,lam_list,mu,alpha,slack,alphagrad,dfnorm,gradnorm
0,"(0,)",0.621044,-0.298695,0.025463,0.113366,"(1.8768983341773733,)","([1.113366094442089],)",1,"(0.0,)","([-0.011133660944420889],)","(tensor(0.4072),)","(0.3711100694729554,)","([[tensor(0.1250, dtype=torch.float64), tensor..."
1,"(1,)",0.59906,-0.305699,0.023148,0.084987,"(1.8771505328445253,)","([1.113366094442089],)",1,"(-0.004072033800184727,)","([0.0],)","(tensor(0.4938),)","(0.399950407229525,)","([[tensor(0.1417, dtype=torch.float64), tensor..."
2,"(2,)",0.6253,-0.298125,0.025463,0.10728,"(1.8775759855590894,)","([1.113366094442089],)",1,"(-0.012267321348190308,)","([0.0],)","(tensor(0.4238),)","(0.4989033262766993,)","([[tensor(0.2227, dtype=torch.float64), tensor..."
3,"(3,)",0.553584,-0.299395,0.013889,0.047519,"(1.8777828748510421,)","([1.113366094442089],)",1,"(-0.023061908781528473,)","([0.0],)","(tensor(0.6449),)","(0.344059665110167,)","([[tensor(0.0620, dtype=torch.float64), tensor..."
4,"(4,)",0.601443,-0.295329,0.027778,0.089329,"(1.8787633240259287,)","([1.113366094442089],)",1,"(-0.038146939128637314,)","([0.0],)","(tensor(0.4120),)","(0.5300749442855338,)","([[tensor(0.2496, dtype=torch.float64), tensor..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,"(95,)",0.384216,-0.196317,0.009259,-0.099081,"(1.8758672263832004,)","([1.113366094442089],)",1.040604,"(-0.24240685999393463,)","([0.0],)","(tensor(-0.0842),)","(0.02409428872350986,)","([[tensor(0.0030, dtype=torch.float64), tensor..."
96,"(96,)",0.362466,-0.199538,0.006944,-0.120665,"(1.8760099829499297,)","([1.113366094442089],)",1.040604,"(-0.24296323955059052,)","([0.0],)","(tensor(0.1280),)","(0.03821160813553693,)","([[tensor(-0.0295, dtype=torch.float64), tenso..."
97,"(97,)",0.37456,-0.194303,0.006944,-0.107998,"(1.8764083735450785,)","([1.113366094442089],)",1.040604,"(-0.24468882381916046,)","([0.0],)","(tensor(-0.0556),)","(0.03191887075859991,)","([[tensor(0.0291, dtype=torch.float64), tensor..."
98,"(98,)",0.358149,-0.200388,0.00463,-0.124286,"(1.8765832006521004,)","([1.113366094442089],)",1.040604,"(-0.2455131858587265,)","([0.0],)","(tensor(-0.0410),)","(0.0135756528361312,)","([[tensor(0.0125, dtype=torch.float64), tensor..."


In [24]:
np.array(result.df["Violations_train"])[-1]

-0.11014268730603398

In [23]:
kappa

-0.01

In [25]:
npt.assert_array_less(np.array(result.df["Violations_train"])[-1], kappa)

In [27]:
result.df["Probability_violations_train"]

0     0.025463
1     0.023148
2     0.025463
3     0.013889
4     0.027778
        ...   
95    0.009259
96    0.006944
97    0.006944
98     0.00463
99    0.006944
Name: Probability_violations_train, Length: 100, dtype: object